# VMGP: Variational Auto-Encoder based Multi-task Genomic Prediction
## Adattamento per Dataset Adaptmap (Classificazione Razze Ovine)

Questo notebook implementa l'architettura **VMGP** descritta nel paper *"VMGP: A unified variational auto-encoder based multi-task model..."*[cite: 10].

### Adattamenti per Adaptmap:
1. **Task:** Classificazione Multi-classe (Razza) invece di Regressione Fenotipica.
2. **Input:** File binari PLINK (`.bed`, `.bim`, `.fam`).
3. **Framework:** PyTorch Lightning per una gestione modulare del training.

---

In [ ]:
# Installazione delle dipendenze necessarie (decommentare se necessario)
# !pip install torch torchvision torchaudio pytorch-lightning pandas-plink scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pandas_plink import read_plink1_bin
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

# Se disponibile, usa torchmetrics per calcolare l'accuratezza in modo efficiente
try:
    import torchmetrics
except ImportError:
    print("Warning: torchmetrics non installato. Le metriche saranno calcolate manualmente.")
    torchmetrics = None

# Imposta seed per riproducibilità
pl.seed_everything(42)

## 1. Definizione dell'Architettura VMGP

Implementazione fedele ai moduli descritti nel paper:
- **Encoder**: Compressione dati genomici[cite: 120].
- **Sampler**: VAE Reparameterization trick[cite: 176].
- **Decoder**: Ricostruzione genomica (Auxiliary Task)[cite: 188].
- **Predictor**: Predizione Fenotipo/Razza (Primary Task)[cite: 190].

In [ ]:
class VMGP_Network(nn.Module):
    def __init__(self, num_snps, num_classes, d1=2048, d2=512, d3=128, latent_dim=128):
        super().__init__()
        
        # --- Encoder [cite: 174] ---
        # Input -> Linear -> BatchNorm -> LeakyReLU -> Dropout
        self.encoder = nn.Sequential(
            nn.Linear(num_snps, d1),
            nn.BatchNorm1d(d1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.1),
            nn.Linear(d1, d2),
            nn.BatchNorm1d(d2),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.1)
        )
        
        # --- Sampler (VAE) [cite: 176] ---
        # Genera vettori media (mu) e log-varianza (sigma)
        self.fc_mu = nn.Linear(d2, latent_dim)
        self.fc_logvar = nn.Linear(d2, latent_dim)
        
        # --- Decoder (Genomic Reconstruction) [cite: 188] ---
        # Speculare all'encoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, d2),
            nn.BatchNorm1d(d2),
            nn.LeakyReLU(0.2),
            nn.Linear(d2, d1),
            nn.BatchNorm1d(d1),
            nn.LeakyReLU(0.2),
            nn.Linear(d1, num_snps) # Output dimensione originale SNP
        )
        
        # --- Predictor (Breed Classification) [cite: 191] ---
        # Latent -> d3 -> Output
        self.predictor = nn.Sequential(
            nn.Linear(latent_dim, d3),
            nn.BatchNorm1d(d3),
            nn.LeakyReLU(0.2),
            nn.Linear(d3, num_classes) # Output logits per le classi di razza
        )

    def reparameterize(self, mu, logvar):
        """
        Reparameterization trick: z = mu + sigma * epsilon
        Permette la backpropagation attraverso il campionamento stocastico[cite: 179].
        """
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        else:
            return mu

    def forward(self, x):
        # 1. Encoding
        encoded = self.encoder(x)
        
        # 2. Sampling Latente
        mu = self.fc_mu(encoded)
        logvar = self.fc_logvar(encoded)
        z = self.reparameterize(mu, logvar)
        
        # 3. Multi-Task Outputs
        x_recon = self.decoder(z)
        y_logits = self.predictor(z)
        
        return x_recon, y_logits, mu, logvar

## 2. Lightning Module (Logica di Training)

Gestisce la Loss Function combinata:
$$ \mathcal{L} = \alpha \times \mathcal{L}_{rec} + (1-\alpha) \times \mathcal{L}_{pre} $$
Dove:
- $\mathcal{L}_{rec}$: MSE + KL Divergence [cite: 211]
- $\mathcal{L}_{pre}$: Cross Entropy (adattata per classificazione) [cite: 219]

In [ ]:
class VMGP_LightningSystem(pl.LightningModule):
    def __init__(self, num_snps, num_classes, alpha=0.5, lr=0.0001):
        super().__init__()
        self.save_hyperparameters()
        self.model = VMGP_Network(num_snps, num_classes)
        self.alpha = alpha

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        # Learning rate 0.0001 come da paper [cite: 224]
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

    def _common_step(self, batch, batch_idx, stage):
        x, y = batch
        x_recon, y_logits, mu, logvar = self(x)
        
        # --- Loss Ricostruzione (MSE + KL) ---
        mse_loss = F.mse_loss(x_recon, x, reduction='mean')
        # KL Divergence: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
        kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        kl_loss /= x.size(0) # Normalizzazione per batch
        
        # Combinazione reconstruction loss (il peso 0.0001 per KL è standard nei VAE)
        loss_rec = mse_loss + 0.0001 * kl_loss
        
        # --- Loss Predizione (CrossEntropy) ---
        loss_pre = F.cross_entropy(y_logits, y)
        
        # --- Total Loss [cite: 203] ---
        total_loss = self.alpha * loss_rec + (1 - self.alpha) * loss_pre
        
        # Calcolo accuratezza
        preds = torch.argmax(y_logits, dim=1)
        acc = (preds == y).float().mean()
        
        # Logging
        self.log(f'{stage}_loss', total_loss, prog_bar=True)
        self.log(f'{stage}_acc', acc, prog_bar=True)
        self.log(f'{stage}_rec_loss', loss_rec, prog_bar=False)
        
        return total_loss

    def training_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, 'train')

    def validation_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, 'val')
    
    def test_step(self, batch, batch_idx):
        return self._common_step(batch, batch_idx, 'test')

## 3. Data Module: PLINK Loading e Quality Control

Carica i file `.bed`, `.bim`, `.fam` ed esegue il preprocessing standard per genomica:
1.  Rimozione SNP con troppi missing values (GENO).
2.  Rimozione SNP con Minor Allele Frequency bassa (MAF).
3.  Imputazione valori mancanti.

In [ ]:
from bed_reader import open_bed

class GenomicDataModule(pl.LightningDataModule):
    """DataModule per dati genetici PLINK con QC e LD pruning"""
    
    def __init__(self, bed_path, batch_size=32, 
                 maf_thresh=0.05, geno_thresh=0.1, min_samples_per_class=10,
                 ld_pruning=True, ld_threshold=0.2, ld_window=50,
                 val_split=0.15, test_split=0.15, coarse_mapping=None):
        super().__init__()
        self.bed_path = bed_path
        self.batch_size = batch_size
        self.maf_thresh = maf_thresh
        self.geno_thresh = geno_thresh
        self.min_samples = min_samples_per_class 
        
        # Parametri LD Pruning
        self.ld_pruning = ld_pruning
        self.ld_threshold = ld_threshold
        self.ld_window = ld_window
        
        # Split ratios
        self.val_split = val_split
        self.test_split = test_split
        
        # Coarse Grained Mapping
        self.coarse_mapping = coarse_mapping
        
        self.imputer = SimpleImputer(strategy='mean')
        self.le = LabelEncoder()
        
    def setup(self, stage=None):
        if hasattr(self, 'train_dataset'):
            return

        print(f"\n{'='*60}")
        print(f"CARICAMENTO DATASET PLINK: {self.bed_path}")
        print(f"{'='*60}")
        
        if not os.path.exists(self.bed_path):
            raise FileNotFoundError(f"File {self.bed_path} non trovato!")

        try:
            # Lettura file PLINK
            G = open_bed(self.bed_path, count_A1=True)
            
            # Lettura FAM per label popolazioni
            fam_path = self.bed_path.replace('.bed', '.fam')
            fam_data = np.loadtxt(fam_path, dtype=str, usecols=0)  # Prima colonna = FID
            y_raw = fam_data
            y_breeds = y_raw.copy() # Save original breeds
            
            # Caricamento genotipi
            # FIX: G.read() restituisce direttamente il numpy array, non un oggetto con attributo .val
            X = G.read()
            
            # Gestione orientamento matrice
            if X.shape[1] == len(y_raw):
                print("⚠️  Trasposizione matrice (SNP×Sample → Sample×SNP)...")
                X = X.T
            
            print(f"📊 Shape iniziale: {X.shape} ({X.shape[0]} campioni, {X.shape[1]} SNP)")

            # --- FILTRO CLASSI RARE (Sempre su y_breeds) ---
            # Filtriamo le razze rare PRIMA del mapping, così rimuoviamo le razze con pochi campioni
            # indipendentemente da come verranno raggruppate.
            unique_classes, counts = np.unique(y_breeds, return_counts=True)
            valid_classes = unique_classes[counts >= self.min_samples]
            mask = np.isin(y_breeds, valid_classes)
            
            n_removed = len(y_breeds) - mask.sum()
            print(f"\n🔍 Filtro Razze Rare (min {self.min_samples} campioni):")
            print(f"   Razze valide: {len(valid_classes)}/{len(unique_classes)}")
            print(f"   Campioni rimossi: {n_removed}")
            
            X = X[mask]
            y_raw = y_raw[mask]
            y_breeds = y_breeds[mask] # Keep aligned

            # --- COARSE GRAINED MAPPING ---
            if self.coarse_mapping:
                print(f"\n🌍 Applicazione Coarse-Grained Mapping...")
                new_y = []
                mask_keep = []
                
                # Flatten mapping for O(1) lookup
                breed_to_category = {}
                for cat, breeds in self.coarse_mapping.items():
                    for breed in breeds:
                        breed_to_category[breed] = cat
                
                for label in y_raw:
                    if label in breed_to_category:
                        new_y.append(breed_to_category[label])
                        mask_keep.append(True)
                    else:
                        mask_keep.append(False)
                
                mask_keep = np.array(mask_keep)
                n_dropped = len(y_raw) - mask_keep.sum()
                
                if n_dropped > 0:
                    print(f"   ⚠️ Rimossi {n_dropped} campioni non presenti nel mapping coarse-grained.")
                    X = X[mask_keep]
                    y_raw = np.array(new_y)
                    y_breeds = y_breeds[mask_keep] # Keep aligned
                else:
                    y_raw = np.array(new_y)
                    
                print(f"   Nuove classi: {np.unique(y_raw)}")
            # ------------------------------------

            # --- QC PIPELINE ---
            print(f"\n🧬 QUALITY CONTROL PIPELINE")
            print(f"{'─'*60}")
            
            # 1. Filtro Missingness SNP
            print("1️⃣  Filtro Missingness SNP...")
            missing_rate_snps = np.isnan(X).mean(axis=0)
            snp_mask = missing_rate_snps < self.geno_thresh
            n_snps_removed = (~snp_mask).sum()
            X = X[:, snp_mask]
            print(f"    ✓ Rimossi {n_snps_removed} SNP con missingness >{self.geno_thresh*100}%")
            
            # 2. Imputazione
            print("2️⃣  Imputazione valori mancanti (media)...")
            X = self.imputer.fit_transform(X)
            print(f"    ✓ Imputazione completata")
            
            # 3. MAF Filter
            print("3️⃣  Filtro MAF (Minor Allele Frequency)...")
            af = np.mean(X, axis=0) / 2.0
            maf = np.minimum(af, 1 - af)
            maf_mask = maf > self.maf_thresh
            n_maf_removed = (~maf_mask).sum()
            X = X[:, maf_mask]
            print(f"    ✓ Rimossi {n_maf_removed} SNP con MAF <{self.maf_thresh}")
            
            # 4. LD Pruning
            if self.ld_pruning:
                X = self._prune_ld(X)

            print(f"\n{'='*60}")
            print(f"📈 DIMENSIONI FINALI: {X.shape}")
            print(f"{'='*60}")

            # Encoding labels
            y = self.le.fit_transform(y_raw)
            self.num_classes = len(self.le.classes_)
            self.num_snps = X.shape[1]
            self.class_names = self.le.classes_
            
            print(f"\n🏷️  Numero Razze: {self.num_classes}")
            print(f"📍 SNP finali: {self.num_snps}")
            
            # SALVATAGGIO DATI PROCESSATI (per K-Fold)
            self.X_processed = X
            self.y_processed = y
            self.y_breeds = y_breeds # Save original breeds
            
            # --- TRAIN/VAL/TEST SPLIT ---
            print(f"\n📂 Creazione Split Dataset...")
            try:
                # Prima split: train vs (val+test)
                test_val_size = self.val_split + self.test_split
                X_train, X_temp, y_train, y_temp = train_test_split(
                    X, y, test_size=test_val_size, stratify=y, random_state=42
                )
                
                # Seconda split: val vs test
                relative_test_size = self.test_split / test_val_size
                X_val, X_test, y_val, y_test = train_test_split(
                    X_temp, y_temp, test_size=relative_test_size, 
                    stratify=y_temp, random_state=42
                )
                
                print(f"   Train: {X_train.shape[0]} campioni")
                print(f"   Val:   {X_val.shape[0]} campioni")
                print(f"   Test:  {X_test.shape[0]} campioni")
                
            except ValueError as e:
                raise ValueError(
                    f"Errore nello split stratificato: {e}\n"
                    f"SOLUZIONE: Aumenta 'min_samples_per_class' (attuale: {self.min_samples})"
                )

            # Creazione TensorDataset
            self.train_dataset = TensorDataset(
                torch.FloatTensor(X_train), 
                torch.LongTensor(y_train)
            )
            self.val_dataset = TensorDataset(
                torch.FloatTensor(X_val), 
                torch.LongTensor(y_val)
            )
            self.test_dataset = TensorDataset(
                torch.FloatTensor(X_test), 
                torch.LongTensor(y_test)
            )
            
            print(f"✅ Setup completato!\n")
            
        except Exception as e:
            raise RuntimeError(f"Errore nel setup DataModule: {e}")

    def balance_dataset(self, X, y, target_samples=50):
        """Bilancia il dataset: upsampling < target, downsampling > target"""
        X_bal_list = []
        y_bal_list = []
        unique_classes = np.unique(y)
        
        for cls in unique_classes:
            mask = y == cls
            X_cls = X[mask]
            y_cls = y[mask]
            n_samples = len(X_cls)
            
            if n_samples < target_samples:
                # Upsample (con replacement)
                indices = np.random.choice(n_samples, target_samples, replace=True)
            elif n_samples > target_samples:
                # Downsample (senza replacement)
                indices = np.random.choice(n_samples, target_samples, replace=False)
            else:
                indices = np.arange(n_samples)
                
            X_bal_list.append(X_cls[indices])
            y_bal_list.append(y_cls[indices])
            
        return np.concatenate(X_bal_list), np.concatenate(y_bal_list)

    def _prune_ld(self, X):
        """LD Pruning con PyTorch per velocità"""
        print(f"4️⃣  LD Pruning (Window={self.ld_window}, r²>{self.ld_threshold})...")
        n_samples, n_snps = X.shape
        
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"    Device: {device}")
        
        # Standardizzazione per correlazione
        X_std = (X - np.mean(X, axis=0)) / (np.std(X, axis=0) + 1e-8)
        X_tensor = torch.tensor(X_std, device=device, dtype=torch.float32)
        
        to_remove = np.zeros(n_snps, dtype=bool)
        step_size = 5
        
        try:
            i = 0
            while i < n_snps:
                if to_remove[i]:
                    i += 1
                    continue
                
                start = i + 1
                end = min(i + 1 + self.ld_window, n_snps)
                
                if start >= end:
                    i += step_size
                    continue
                
                window_indices = [j for j in range(start, end) if not to_remove[j]]
                
                if not window_indices:
                    i += step_size
                    continue
                
                # Correlazione r²
                pivot = X_tensor[:, i].unsqueeze(1)
                targets = X_tensor[:, window_indices]
                corrs = torch.mm(pivot.t(), targets).squeeze(0) / n_samples
                r2 = corrs ** 2
                
                # SNP in alto LD
                bad_rel_idx = torch.nonzero(r2 > self.ld_threshold).squeeze(-1).cpu().numpy()
                
                if bad_rel_idx.ndim == 0 and bad_rel_idx.size == 1:
                    bad_rel_idx = [int(bad_rel_idx)]
                    
                for local_idx in bad_rel_idx:
                    global_idx = window_indices[local_idx]
                    to_remove[global_idx] = True
                    
                i += step_size
                
                if i % 5000 == 0:
                    print(f"    ...{i}/{n_snps} SNP scansionati")

        except RuntimeError as e:
            print(f"⚠️  Errore GPU: {e}. LD Pruning saltato.")
            return X

        mask_keep = ~to_remove
        n_removed = np.sum(to_remove)
        print(f"    ✓ Rimossi {n_removed} SNP ridondanti (LD)")
        
        return X[:, mask_keep]

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset, 
            batch_size=self.batch_size, 
            shuffle=True, 
            num_workers=4,
            pin_memory=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset, 
            batch_size=self.batch_size, 
            num_workers=4,
            pin_memory=True
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset, 
            batch_size=self.batch_size,
            num_workers=4,
            pin_memory=True
        )

In [ ]:
# ⚠️ MODIFICA QUESTO PATH CON IL TUO DATASET ⚠️
BED_PATH = "data/ADAPTmap_genotypeTOP_20160222_full.bed"

# Coarse Grained Mapping
COARSE_MAPPING = {
    'LATTE': ['ALP', 'ARG', 'ASP', 'BIO', 'CRS', 'GGT', 'KLS', 'LNR', 'MLG', 'MLS', 'MLT', 'MUG', 'NIC', 'ORO', 'PTV', 'RME', 'SAA', 'SAR', 'TOG', 'VSS', 'NRW'],
    'FIBRA': ['ANG', 'ANK', 'CAS'],
    'CARNE': ['BOE', 'RAN', 'TED', 'BRI', 'BEY', 'MOX', 'RAS', 'THA']
}

# Hyperparameters
config = {
    # DataModule
    'bed_path': BED_PATH,
    'batch_size': 32,
    'maf_thresh': 0.01,
    'geno_thresh': 0.1,
    'min_samples_per_class': 10,
    'ld_pruning': True,
    'ld_threshold': 0.2,
    'ld_window': 50,
    'val_split': 0.15,
    'test_split': 0.15,
    
    # Balancing
    'target_samples': 50, # Target per il bilanciamento
    
    # Model (VMGP)
    'alpha': 0.5,
    'lr': 0.0001,
    
    # Training
    'max_epochs': 100,
    'accelerator': 'gpu' if torch.cuda.is_available() else 'cpu',
    'devices': 1,
}

print("📋 CONFIGURAZIONE:")
for k, v in config.items():
    print(f"   {k}: {v}")

In [ ]:
import numpy as np
from typing import Tuple, Literal
from sklearn.neighbors import NearestNeighbors

def balance_dataset_smote(X: np.ndarray, y: np.ndarray, target_samples: int = 50, k_neighbors: int = 5) -> Tuple[np.ndarray, np.ndarray]:
    """
    SMOTE-like interpolation adapted for genomic SNP data.
    
    For minority classes: Generate synthetic samples by interpolating between neighbors.
    For majority classes: Random downsampling.
    
    SNP values are rounded to 0, 1, or 2 after interpolation.
    
    Args:
        X: Feature matrix (n_samples, n_features) with SNP values 0, 1, 2
        y: Labels array
        target_samples: Target samples per class
        k_neighbors: Number of neighbors for SMOTE interpolation
    
    Returns:
        Balanced X and y arrays
    """
    X_bal_list = []
    y_bal_list = []
    unique_classes = np.unique(y)
    
    for cls in unique_classes:
        mask = y == cls
        X_cls = X[mask]
        y_cls = y[mask]
        n_samples = len(X_cls)
        
        if n_samples >= target_samples:
            # Downsample
            indices = np.random.choice(n_samples, target_samples, replace=False)
            X_bal_list.append(X_cls[indices])
            y_bal_list.append(y_cls[indices])
        else:
            # SMOTE-like oversampling
            n_synthetic = target_samples - n_samples
            
            # Use min of k_neighbors and available samples - 1
            k = min(k_neighbors, n_samples - 1)
            if k < 1:
                # Not enough samples for SMOTE, fall back to random
                indices = np.random.choice(n_samples, target_samples, replace=True)
                X_bal_list.append(X_cls[indices])
                y_bal_list.append(y_cls[indices])
                continue
            
            # Fit nearest neighbors
            nn = NearestNeighbors(n_neighbors=k + 1)
            nn.fit(X_cls)
            
            # Generate synthetic samples
            synthetic_samples = []
            for _ in range(n_synthetic):
                # Pick random sample
                idx = np.random.randint(n_samples)
                sample = X_cls[idx]
                
                # Get neighbors
                _, neighbor_indices = nn.kneighbors([sample])
                neighbor_idx = neighbor_indices[0, np.random.randint(1, k + 1)]
                neighbor = X_cls[neighbor_idx]
                
                # Interpolate
                alpha = np.random.random()
                synthetic = sample + alpha * (neighbor - sample)
                
                # Round to valid SNP values (0, 1, 2)
                synthetic = np.clip(np.round(synthetic), 0, 2)
                synthetic_samples.append(synthetic)
            
            # Combine original + synthetic
            X_combined = np.vstack([X_cls, np.array(synthetic_samples)])
            y_combined = np.full(len(X_combined), cls)
            
            X_bal_list.append(X_combined)
            y_bal_list.append(y_combined)
    
    return np.concatenate(X_bal_list), np.concatenate(y_bal_list)

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, silhouette_score, davies_bouldin_score, classification_report, mean_squared_error
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist
import pandas as pd
import numpy as np
import shutil
import torch
import os
import re
import gc

# --- HELPER FUNCTIONS (LEGACY & NEW) ---

def compute_local_structure(embeddings_train, labels_train):
    """Local Structure (L) - 3NN accuracy on training set"""
    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(embeddings_train, labels_train)
    preds = knn.predict(embeddings_train)
    accuracy = accuracy_score(labels_train, preds)
    return accuracy


def compute_generalization(embeddings_train, labels_train, embeddings_val, labels_val):
    """Generalization (GE) - 3NN accuracy on validation set (trained on training set)"""
    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(embeddings_train, labels_train)
    preds = knn.predict(embeddings_val)
    accuracy = accuracy_score(labels_val, preds)
    return accuracy


def compute_neighbor_overlap(embeddings, genotypes, k_values=[3, 10, 30], max_samples=1000):
    """
    Overlap between k-nearest neighbors in genotype space vs embedding space.
    Uses a subset of samples if dataset is large to save time/memory.
    """
    n_samples = len(embeddings)
    if n_samples > max_samples:
        idx = np.random.choice(n_samples, max_samples, replace=False)
        embeddings = embeddings[idx]
        genotypes = genotypes[idx]
    
    # Distance matricess
    # Use Euclidean for both as approximation (or cityblock for genotypes if preferred/fast enough)
    dist_geno = cdist(genotypes, genotypes, metric='cityblock') 
    dist_emb = cdist(embeddings, embeddings, metric='euclidean')
    
    overlaps = {}
    for k in k_values:
        overlap_sum = 0
        for i in range(len(embeddings)):
            # k-NN indices (excluding self at index 0)
            nn_geno = np.argsort(dist_geno[i])[1:k+1]
            nn_emb = np.argsort(dist_emb[i])[1:k+1]s
            
            # Intersection
            overlap = len(np.intersect1d(nn_geno, nn_emb)) / k
            overlap_sum += overlap
        
        overlaps[f'NO_k{k}'] = overlap_sum / len(embeddings)
    
    return overlaps


def run_experiment(name, balanced=False, coarse_mapping=None, max_epochs=50, k_folds=2):
    print(f"\n{'='*80}")
    print(f"🧪 EXPERIMENT: {name}")
    print(f"   Balanced: {balanced}")
    print(f"   Coarse Grained: {coarse_mapping is not None}")
    print(f"   K-Fold CV: {k_folds} folds")
    print(f"{'='*80}\n")
    
    # 1. Setup DataModule (just to get processed data)
    dm = GenomicDataModule(
        bed_path=config['bed_path'],
        batch_size=config['batch_size'],
        maf_thresh=config['maf_thresh'],
        geno_thresh=config['geno_thresh'],
        min_samples_per_class=config['min_samples_per_class'],
        ld_pruning=config['ld_pruning'],
        ld_threshold=config['ld_threshold'],
        ld_window=config['ld_window'],
        val_split=config['val_split'],
        test_split=config['test_split'],
        coarse_mapping=coarse_mapping
    )
    
    dm.setup()
    
    # 2. Get processed data for K-Fold
    if not hasattr(dm, 'X_processed'):
        raise ValueError("DataModule doesn't have 'X_processed'. Run setup() first.")
    
    X = dm.X_processed
    y = dm.y_processed
    y_breeds = getattr(dm, 'y_breeds', None)
    
    # 3. K-Fold Cross Validation
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    fold_metrics = {
        'Local Structure (L)': [],
        'Generalization (GE)': [],
        'Silhouette': [],
        'Davies-Bouldin': [],
        'Reconstruction MSE': [],
        'Predictor Accuracy': [],
        'NO_k3': [],
        'NO_k10': [],
        'NO_k30': []
    }
    
        
    # --- MODIFICA: Bilanciamento GLOBALE prima del K-Fold ---
    if balanced:
        print(f"   ⚖️ Balancing dataset globally (SMOTE)...")
        
        if coarse_mapping and y_breeds is not None:
             print(f"   ℹ️ Balancing by BREED (50 samples each) then mapping to COARSE...")
             # Balance using breeds
             X, y_breeds_bal = balance_dataset_smote(X, y_breeds, config['target_samples'])
             
             # Map balanced breeds back to coarse labels (y)
             # Re-create mapping from breed to coarse label
             breed_to_category = {}
             for cat, breeds in coarse_mapping.items():
                 for breed in breeds:
                     breed_to_category[breed] = cat
             
             y_bal_labels = []
             for breed in y_breeds_bal:
                 y_bal_labels.append(breed_to_category[breed])
             
             y_bal_labels = np.array(y_bal_labels)
             
             # Encode labels using the LabelEncoder from dm
             y = dm.le.transform(y_bal_labels)
             
        else:
            # Assicurati che balance_dataset_smote sia importato o disponibile
            X, y = balance_dataset_smote(X, y, config['target_samples'])
            
        print(f"   Balanced Dataset: {len(y)} samples")
        
    best_fold_idx = None
    best_fold_acc = -1
    best_fold_emb_val = None  # Embeddings del validation set (inferenza)
    best_fold_lbl_val = None
    best_fold_path = None # Store path to best checkpoint
    
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n🔹 FOLD {fold+1}/{k_folds}")
        
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        

        # Create datasets and loaders
        train_ds = TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train))
        val_ds = TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val))
        
        train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=config['batch_size'], num_workers=4, pin_memory=True)
        
        # Initialize model for this fold
        model_fold = VMGP_LightningSystem(
            num_snps=dm.num_snps,
            num_classes=dm.num_classes,
            alpha=config['alpha'],
            lr=config['lr']
        )
        
        # Callbacks
        checkpoint_callback = ModelCheckpoint(
            dirpath=f'checkpoints_vae/{name.replace(" ", "_")}/fold_{fold+1}',
            filename='{epoch:02d}-{val_loss:.4f}',
            save_top_k=1,
            monitor='val_loss',
            mode='min'
        )
        
        early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min', verbose=False)
        
        # Trainer
        trainer = pl.Trainer(
            precision="16-mixed",
            max_epochs=max_epochs,
            accelerator=config['accelerator'],
            devices=config['devices'],
            callbacks=[checkpoint_callback, early_stop],
            enable_progress_bar=False,
            enable_model_summary=False,
            logger=False,
            check_val_every_n_epoch=1
        )
        
    
        # Train fold
        trainer.fit(model_fold, train_loader, val_loader)
        
        # Load best model for this fold if available
        best_model_path = checkpoint_callback.best_model_path
        if best_model_path:
            try:
                model_fold = VMGP_LightningSystem.load_from_checkpoint(best_model_path)
            except Exception as e:
                print(f"   ⚠️ Failed loading best model {best_model_path}: {e}")
       
        model_fold.eval()
        model_fold.freeze()
        
        # --- Evaluation for this fold ---
        
        # Extract Train Data
        emb_train = []
        lbl_train = []
        snps_train = []
        recon_train = []
        
        with torch.no_grad():
            for batch in train_loader:
                snps, y_batch = batch
                snps = snps.to(model_fold.device)
                x_recon, _, mu, _ = model_fold(snps)
                
                emb_train.append(mu.cpu().numpy())
                lbl_train.append(y_batch.cpu().numpy())
                snps_train.append(snps.cpu().numpy())
                recon_train.append(x_recon.cpu().numpy())
                
        emb_train = np.concatenate(emb_train)
        lbl_train = np.concatenate(lbl_train)
        snps_train = np.concatenate(snps_train)
        recon_train = np.concatenate(recon_train)
        
        # Extract Val Data
        emb_val = []
        lbl_val = []
        snps_val = []
        recon_val = []
        
        with torch.no_grad():
            for batch in val_loader:
                snps, y_batch = batch
                snps = snps.to(model_fold.device)
                x_recon, _, mu, _ = model_fold(snps)
                
                emb_val.append(mu.cpu().numpy())
                lbl_val.append(y_batch.cpu().numpy())
                snps_val.append(snps.cpu().numpy())
                recon_val.append(x_recon.cpu().numpy())
                
        emb_val = np.concatenate(emb_val)
        lbl_val = np.concatenate(lbl_val)
        snps_val = np.concatenate(snps_val)
        recon_val = np.concatenate(recon_val)
        
        # Calculate Metrics
        L = compute_local_structure(emb_train, lbl_train)
        GE = compute_generalization(emb_train, lbl_train, emb_val, lbl_val)
        no_metrics = compute_neighbor_overlap(emb_train, snps_train, k_values=[3, 10, 30], max_samples=1000)
        mse_val = mean_squared_error(snps_val, recon_val)
        
        # Predictor Accuracy
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                snps, y_batch = batch
                snps = snps.to(model_fold.device)
                _, y_logits, _, _ = model_fold(snps)
                preds = torch.argmax(y_logits, dim=1)
                correct += (preds.cpu() == y_batch.cpu()).sum().item()
                total += y_batch.size(0)
        acc_predictor = correct / total
        
        sil = silhouette_score(emb_train, lbl_train) if len(np.unique(lbl_train)) > 1 else 0
        db = davies_bouldin_score(emb_train, lbl_train) if len(np.unique(lbl_train)) > 1 else 0
        
        # Store metrics
        fold_metrics['Local Structure (L)'].append(L)
        fold_metrics['Generalization (GE)'].append(GE)
        fold_metrics['Silhouette'].append(sil)
        fold_metrics['Davies-Bouldin'].append(db)
        fold_metrics['Reconstruction MSE'].append(mse_val)
        fold_metrics['Predictor Accuracy'].append(acc_predictor)
        fold_metrics['NO_k3'].append(no_metrics['NO_k3'])
        fold_metrics['NO_k10'].append(no_metrics['NO_k10'])
        fold_metrics['NO_k30'].append(no_metrics['NO_k30'])
        
        print(f"   ✅ Fold {fold+1} | GE: {GE:.4f} | Sil: {sil:.4f} | MSE: {mse_val:.4f} | Acc: {acc_predictor:.4f}")
        
        # Track best fold (based on Generalization)
        if GE > best_fold_acc:
            best_fold_acc = GE
            best_fold_idx = fold
            best_fold_emb_val = emb_val.copy()  # Salva embeddings validation (inferenza)
            best_fold_lbl_val = lbl_val.copy()
            best_fold_path = best_model_path # Save the path to the best checkpoint
        
        # --- CLEANUP MEMORY ---
        del model_fold
        del trainer
        del train_loader
        del val_loader
        del train_ds
        del val_ds
        
        # Release large numpy arrays
        del snps_train
        del recon_train
        del snps_val
        del recon_val
        del emb_train
        del lbl_train
        del emb_val
        del lbl_val
        
        torch.cuda.empty_cache()
        gc.collect()
            
    # Save the best model checkpoint for this experiment
    exp_checkpoint_dir = f'checkpoints_vae/{name.replace(" ", "_")}'
    os.makedirs(exp_checkpoint_dir, exist_ok=True)
    best_checkpoint_path = os.path.join(exp_checkpoint_dir, 'best_model.ckpt')
    
    if best_fold_path and os.path.exists(best_fold_path):
        shutil.copy(best_fold_path, best_checkpoint_path)
        print(f"\n💾 Saved best checkpoint from Fold {best_fold_idx+1}: {best_checkpoint_path}")
    else:
        print(f"\n⚠️ Could not find best checkpoint to save.")

    # 4. Aggregate Results
    print(f"\n{'='*80}")
    print(f"📊 K-FOLD RESULTS (Mean ± Std):")
    for metric, values in fold_metrics.items():
        print(f"   {metric}: {np.mean(values):.4f} ± {np.std(values):.4f}")
    print(f"   Best Fold:       {best_fold_idx + 1} (GE: {best_fold_acc:.4f})")
    print(f"{'='*80}\n")
    
    results = {
        'Experiment': name,
        'Best Fold': best_fold_idx + 1,
        'Best Fold GE': best_fold_acc,
        'Num Classes': dm.num_classes,
        'Num SNPs': dm.num_snps,
        'K Folds': k_folds
    }
    
    for metric, values in fold_metrics.items():
        results[f'{metric} (mean)'] = np.mean(values)
        results[f'{metric} (std)'] = np.std(values)
    
    # 5. Visualization (using best fold VALIDATION embeddings - inference)
    print(f"📈 Creating PCA visualization for best fold ({best_fold_idx + 1}) - Validation Set (Inference)...")
    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(best_fold_emb_val)
    
    plt.figure(figsize=(10, 8))
    unique_labels = np.unique(best_fold_lbl_val)
    colors = plt.cm.tab20(np.linspace(0, 1, len(unique_labels)))
    
    for i, cls_idx in enumerate(unique_labels):
        mask = best_fold_lbl_val == cls_idx
        plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=dm.class_names[cls_idx], alpha=0.6, s=20)
    
    plt.title(f'{name} - Best Fold {best_fold_idx+1} INFERENCE (GE: {best_fold_acc:.3f})')
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2, fontsize='small')
    plt.tight_layout()
    plt.savefig(f'plot_vae_{name.replace(" ", "_")}_kfold.png')
    plt.show()
    
    return results

In [ ]:
# --- RUN EXPERIMENTS ---

all_results = []

# 1. Original (Fine-Grained, Unbalanced)
res1 = run_experiment("Original Unbalanced", balanced=False, coarse_mapping=None, max_epochs=config['max_epochs'])
all_results.append(res1)

# 2. Balanced (Fine-Grained, Balanced)
res2 = run_experiment("Original Balanced", balanced=True, coarse_mapping=None, max_epochs=config['max_epochs'])
all_results.append(res2)

# 3. Coarse (Coarse-Grained, Unbalanced)
res3 = run_experiment("Coarse Unbalanced", balanced=False, coarse_mapping=COARSE_MAPPING, max_epochs=config['max_epochs'])
all_results.append(res3)

# 4. Coarse Balanced (Coarse-Grained, Balanced)
res4 = run_experiment("Coarse Balanced", balanced=True, coarse_mapping=COARSE_MAPPING, max_epochs=config['max_epochs'])
all_results.append(res4)

# --- COMPARISON TABLE ---
df_results = pd.DataFrame(all_results)
print("\n\n🏆 FINAL COMPARISON (VAE):")
# Reorder columns for better readability
cols = ['Experiment', 'Local Structure (L)', 'Generalization (GE)', 'Predictor Accuracy', 'Reconstruction MSE', 'Silhouette', 'Davies-Bouldin', 'NO_k3', 'NO_k10', 'NO_k30']
# Add remaining columns
cols += [c for c in df_results.columns if c not in cols]
df_results = df_results[cols]

print(df_results)

# Save results
df_results.to_csv("experiment_comparison_vae.csv", index=False)
print("✅ Results saved to experiment_comparison_vae.csv")

In [ ]:
# --- ANALISI APPROFONDITA: MATRICI DI CONFUSIONE E METRICHE INTUITIVE ---
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay, accuracy_score, silhouette_score, davies_bouldin_score, mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import torch
import os
import pandas as pd

def analyze_model_performance(experiment_name, config):
    print(f"\n{'='*80}")
    print(f"🔬 ANALISI PERFORMANCE APPROFONDITA: {experiment_name}")
    print(f"{'='*80}\n")
    
    # 1. Caricamento Modello
    checkpoint_path = f"checkpoints_vae/{experiment_name.replace(' ', '_')}/best_model.ckpt"
    if not os.path.exists(checkpoint_path):
        print(f"⚠️ Checkpoint non trovato: {checkpoint_path}")
        return None

    print(f"   Caricamento modello da: {checkpoint_path}")
    try:
        model = VMGP_LightningSystem.load_from_checkpoint(checkpoint_path)
    except Exception as e:
        print(f"Errore caricamento modello: {e}")
        return None
        
    model.eval()
    model.freeze()
    
    # 2. Caricamento Dati (Train & Validation)
    use_coarse = "Coarse" in experiment_name
    mapping = COARSE_MAPPING if use_coarse else None
    
    dm_viz = GenomicDataModule(
        bed_path=config['bed_path'],
        batch_size=config['batch_size'],
        maf_thresh=config['maf_thresh'],
        geno_thresh=config['geno_thresh'],
        min_samples_per_class=config['min_samples_per_class'],
        ld_pruning=config['ld_pruning'],
        ld_threshold=config['ld_threshold'],
        ld_window=config['ld_window'],
        val_split=config['val_split'],
        test_split=config['test_split'],
        coarse_mapping=mapping
    )
    dm_viz.setup()
    train_loader = dm_viz.train_dataloader()
    val_loader = dm_viz.val_dataloader()
    
    # 3. Estrazione Embeddings e Predizioni
    
    # --- TRAIN SET (per metriche di struttura e training KNN) ---
    print("   Estrazione feature Training Set...")
    emb_train = []
    lbl_train = []
    snps_train = []
    
    with torch.no_grad():
        for batch in train_loader:
            x, y = batch
            x = x.to(model.device)
            _, _, mu, _ = model(x)
            
            emb_train.append(mu.cpu().numpy())
            lbl_train.append(y.cpu().numpy())
            snps_train.append(x.cpu().numpy())
            
    emb_train = np.concatenate(emb_train)
    lbl_train = np.concatenate(lbl_train)
    snps_train = np.concatenate(snps_train)
    
    # --- VAL SET (per metriche di generalizzazione e accuratezza) ---
    print("   Estrazione feature Validation Set...")
    emb_val = []
    lbl_val = []
    snps_val = []
    recon_val = []
    preds_val = []
    
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch
            x = x.to(model.device)
            x_recon, y_logits, mu, _ = model(x)
            
            emb_val.append(mu.cpu().numpy())
            lbl_val.append(y.cpu().numpy())
            snps_val.append(x.cpu().numpy())
            recon_val.append(x_recon.cpu().numpy())
            preds_val.append(torch.argmax(y_logits, dim=1).cpu().numpy())
            
    emb_val = np.concatenate(emb_val)
    lbl_val = np.concatenate(lbl_val)
    snps_val = np.concatenate(snps_val)
    recon_val = np.concatenate(recon_val)
    preds_val = np.concatenate(preds_val)
    
    # 4. Calcolo Metriche Complete
    print("\n📊 CALCOLO METRICHE...")
    
    # A. Metriche Latent Space
    L = compute_local_structure(emb_train, lbl_train)
    GE = compute_generalization(emb_train, lbl_train, emb_val, lbl_val)
    sil = silhouette_score(emb_train, lbl_train) if len(np.unique(lbl_train)) > 1 else 0
    db = davies_bouldin_score(emb_train, lbl_train) if len(np.unique(lbl_train)) > 1 else 0
    
    # B. Neighbor Overlap
    no_metrics = compute_neighbor_overlap(emb_train, snps_train, k_values=[3, 10, 30], max_samples=1000)
    
    # C. Metriche Ricostruzione
    mse_val = mean_squared_error(snps_val, recon_val)
    
    # Genotype Accuracy (Intuitiva)
    x_true_int = np.clip(np.round(snps_val), 0, 2).astype(int).flatten()
    x_recon_int = np.clip(np.round(recon_val), 0, 2).astype(int).flatten()
    genotype_acc = np.mean(x_true_int == x_recon_int)
    
    # D. Metriche Predittore
    acc_predictor = accuracy_score(lbl_val, preds_val)
    
    # Dizionario Risultati
    results_dict = {
        'Experiment': experiment_name,
        'Local Structure (L)': L,
        'Generalization (GE)': GE,
        'Predictor Accuracy': acc_predictor,
        'Genotype Accuracy': genotype_acc,
        'Reconstruction MSE': mse_val,
        'Silhouette': sil,
        'Davies-Bouldin': db,
        'NO_k3': no_metrics['NO_k3'],
        'NO_k10': no_metrics['NO_k10'],
        'NO_k30': no_metrics['NO_k30']
    }
    
    # 5. Visualizzazioni
    
    # --- A. MATRICE DI CONFUSIONE CLASSIFICAZIONE (PRIME 30 CLASSI) ---
    n_classes = len(dm_viz.class_names)
    n_show = min(30, n_classes) # Mostra massimo 30 classi
    
    print(f"\n📉 Visualizzazione Matrice di Confusione (Prime {n_show}/{n_classes} classi)...")
    
    # Calcolo su TUTTE le classi per avere la normalizzazione corretta
    cm = confusion_matrix(lbl_val, preds_val)
    
    # Normalizziamo per riga (Recall)
    cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)
    
    # Selezioniamo solo la sottomatrice in alto a sinistra
    cm_subset = cm_norm[:n_show, :n_show]
    subset_classes = dm_viz.class_names[:n_show]
    
    # Ingrandiamo la figura per rendere le celle più grandi e leggibili
    # 0.8 pollici per cella
    figsize_dim = n_show * 0.8
    plt.figure(figsize=(figsize_dim, figsize_dim * 0.8))
    
    annot_labels = np.where(cm_subset > 0.01, np.char.mod('%.2f', cm_subset), "")
    annot_size = 12 # Font più grande
    
    sns.heatmap(cm_subset, annot=annot_labels, fmt='', cmap='Blues', 
                xticklabels=subset_classes, yticklabels=subset_classes,
                linewidths=1.0, linecolor='lightgray', square=True,
                cbar_kws={"shrink": .8, "label": "Recall (Normalized by Row)"}, 
                annot_kws={"size": annot_size, "weight": "bold"})
                
    plt.title(f'Confusion Matrix (First {n_show} Classes) - {experiment_name}', fontsize=18, pad=20)
    plt.ylabel('True Label', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(rotation=0, fontsize=12)
    plt.tight_layout()
    
    # Salvataggio PNG
    filename_cm = f"confusion_matrix_{experiment_name.replace(' ', '_')}.png"
    plt.savefig(filename_cm, dpi=300, bbox_inches='tight')
    print(f"   💾 Saved confusion matrix to {filename_cm}")
    plt.show()
    
    # --- B. MATRICE DI CONFUSIONE RICOSTRUZIONE (GENOTIPI 0, 1, 2) ---
    # Campionamento per velocità
    if len(x_true_int) > 1_000_000:
        idx = np.random.choice(len(x_true_int), 1_000_000, replace=False)
        x_true_sample = x_true_int[idx]
        x_recon_sample = x_recon_int[idx]
    else:
        x_true_sample = x_true_int
        x_recon_sample = x_recon_int
        
    cm_geno = confusion_matrix(x_true_sample, x_recon_sample, labels=[0, 1, 2])
    # Normalizziamo anche questa per leggibilità
    cm_geno_norm = cm_geno.astype('float') / cm_geno.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_geno_norm, annot=True, fmt='.2%', cmap='Greens', 
                xticklabels=['0 (AA)', '1 (AB)', '2 (BB)'], 
                yticklabels=['0 (AA)', '1 (AB)', '2 (BB)'])
    plt.title(f'Genotype Reconstruction Confusion Matrix (Normalized)\n(Sampled {len(x_true_sample)} SNPs)')
    plt.ylabel('True Genotype')
    plt.xlabel('Reconstructed Genotype (Rounded)')
    plt.tight_layout()
    plt.show()
    
    print(f"\n🧬 METRICA INTUITIVA RICOSTRUZIONE:")
    print(f"   Genotype Accuracy (Accuratezza Genotipica): {genotype_acc*100:.2f}%")
    
    return results_dict

# Esegui per tutti gli esperimenti e raccogli i risultati
all_experiment_results = []
experiments = ["Original Unbalanced", "Original Balanced", "Coarse Unbalanced", "Coarse Balanced"]

for exp in experiments:
    res = analyze_model_performance(exp, config)
    if res:
        all_experiment_results.append(res)

# Salvataggio CSV Finale
if all_experiment_results:
    final_df = pd.DataFrame(all_experiment_results)
    # Riordina colonne per leggibilità
    cols = ['Experiment', 'Local Structure (L)', 'Generalization (GE)', 'Predictor Accuracy', 
            'Genotype Accuracy', 'Reconstruction MSE', 'Silhouette', 'Davies-Bouldin', 
            'NO_k3', 'NO_k10', 'NO_k30']
    final_df = final_df[cols]
    
    final_df.to_csv("final_results_vae_detailed.csv", index=False)
    print("\n\n✅ Tutti i risultati salvati in final_results_vae_detailed.csv")
    print(final_df)